In [14]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/"
    "GoogleAI_contest/Jaehwang/3class/lstm_training"
)

CODE_DIR = PROJECT_DIR / "code"
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

PREPROCESS_SCRIPT = (
    CODE_DIR / "preprocess_3class_train_holdout.py"
)

TRAIN_ACTIVITY_PATH = RAW_DIR / "train_activity.csv"
TRAIN_SLEEP_PATH = RAW_DIR / "train_sleep.csv"
TRAIN_LABEL_PATH = RAW_DIR / "training_label.csv"
TRAIN_MMSE_PATH = RAW_DIR / "train_mmse.csv"

HOLDOUT_ACTIVITY_PATH = RAW_DIR / "val_activity.csv"
HOLDOUT_SLEEP_PATH = RAW_DIR / "val_sleep.csv"
HOLDOUT_LABEL_PATH = RAW_DIR / "val_label.csv"
HOLDOUT_MMSE_PATH = RAW_DIR / "val_mmse.csv"

DATASET_PATH = (
    PROCESSED_DIR / "lstm_3class_train_holdout.pkl"
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [15]:
paths = {
    "preprocess": PREPROCESS_SCRIPT,
    "train activity": TRAIN_ACTIVITY_PATH,
    "train sleep": TRAIN_SLEEP_PATH,
    "train label": TRAIN_LABEL_PATH,
    "train mmse": TRAIN_MMSE_PATH,
    "holdout activity": HOLDOUT_ACTIVITY_PATH,
    "holdout sleep": HOLDOUT_SLEEP_PATH,
    "holdout label": HOLDOUT_LABEL_PATH,
    "holdout mmse": HOLDOUT_MMSE_PATH,
}

for name, path in paths.items():
    print(
        f"{name:20s}",
        f"exists={path.exists()}",
        path,
    )

preprocess           exists=True /content/drive/MyDrive/GoogleAI_contest/Jaehwang/3class/lstm_training/code/preprocess_3class_train_holdout.py
train activity       exists=True /content/drive/MyDrive/GoogleAI_contest/Jaehwang/3class/lstm_training/data/raw/train_activity.csv
train sleep          exists=True /content/drive/MyDrive/GoogleAI_contest/Jaehwang/3class/lstm_training/data/raw/train_sleep.csv
train label          exists=True /content/drive/MyDrive/GoogleAI_contest/Jaehwang/3class/lstm_training/data/raw/training_label.csv
train mmse           exists=True /content/drive/MyDrive/GoogleAI_contest/Jaehwang/3class/lstm_training/data/raw/train_mmse.csv
holdout activity     exists=True /content/drive/MyDrive/GoogleAI_contest/Jaehwang/3class/lstm_training/data/raw/val_activity.csv
holdout sleep        exists=True /content/drive/MyDrive/GoogleAI_contest/Jaehwang/3class/lstm_training/data/raw/val_sleep.csv
holdout label        exists=True /content/drive/MyDrive/GoogleAI_contest/Jaehwang/3cl

In [16]:
!python "{PREPROCESS_SCRIPT}" \
  --train-activity "{TRAIN_ACTIVITY_PATH}" \
  --train-sleep "{TRAIN_SLEEP_PATH}" \
  --train-label "{TRAIN_LABEL_PATH}" \
  --train-mmse "{TRAIN_MMSE_PATH}" \
  --holdout-activity "{HOLDOUT_ACTIVITY_PATH}" \
  --holdout-sleep "{HOLDOUT_SLEEP_PATH}" \
  --holdout-label "{HOLDOUT_LABEL_PATH}" \
  --holdout-mmse "{HOLDOUT_MMSE_PATH}" \
  --output "{DATASET_PATH}" \
  --audit-dir "{PROCESSED_DIR}"

3-class train/holdout preprocessing completed

[TRAIN]
X: (5500, 7, 47)
patients: 138
patient distribution: {'Normal': 84, 'MCI': 46, 'Dementia': 8}
window distribution: {'Normal': 3359, 'MCI': 1900, 'Dementia': 241}
excluded patients: [{'patient_id': 'nia+088@rowan.kr', 'class_name': 'MCI', 'merged_days': 42}, {'patient_id': 'nia+219@rowan.kr', 'class_name': 'Dementia', 'merged_days': 44}, {'patient_id': 'nia+229@rowan.kr', 'class_name': 'Normal', 'merged_days': 60}]

[HOLDOUT]
X: (1440, 7, 47)
patients: 32
patient distribution: {'Normal': 26, 'MCI': 4, 'Dementia': 2}
window distribution: {'Normal': 1118, 'MCI': 247, 'Dementia': 75}
excluded patients: [{'patient_id': 'nia+112@rowan.kr', 'class_name': 'Dementia', 'merged_days': 40}]

[TOTAL]
windows: 6940
patients with windows: 170
train-holdout patient overlap: 0

Saved dataset: /content/drive/MyDrive/GoogleAI_contest/Jaehwang/3class/lstm_training/data/processed/lstm_3class_train_holdout.pkl
Saved audit: /content/drive/MyDrive/GoogleA

In [17]:
import pickle
import numpy as np
import pandas as pd

with DATASET_PATH.open("rb") as f:
    dataset = pickle.load(f)

print("Train X:", dataset["train"]["X"].shape)
print("Train y:", dataset["train"]["y"].shape)

print("Holdout X:", dataset["holdout"]["X"].shape)
print("Holdout y:", dataset["holdout"]["y"].shape)

print("\nTrain window labels:")
print(
    np.unique(
        dataset["train"]["y"],
        return_counts=True,
    )
)

print("\nHoldout window labels:")
print(
    np.unique(
        dataset["holdout"]["y"],
        return_counts=True,
    )
)

Train X: (5500, 7, 47)
Train y: (5500,)
Holdout X: (1440, 7, 47)
Holdout y: (1440,)

Train window labels:
(array([0, 1, 2]), array([3359, 1900,  241]))

Holdout window labels:
(array([0, 1, 2]), array([1118,  247,   75]))


In [18]:
import pickle
import numpy as np
import pandas as pd

with DATASET_PATH.open("rb") as f:
    dataset = pickle.load(f)

train = dataset["train"]
holdout = dataset["holdout"]

print("Train X:", train["X"].shape)
print("Holdout X:", holdout["X"].shape)
print("Feature 수:", len(dataset["feature_names"]))

train_subjects = pd.DataFrame({
    "patient_id": train["patient_id"],
    "y": train["y"],
}).drop_duplicates("patient_id")

holdout_subjects = pd.DataFrame({
    "patient_id": holdout["patient_id"],
    "y": holdout["y"],
}).drop_duplicates("patient_id")

print("\nTrain 환자 분포")
print(train_subjects["y"].value_counts().sort_index())

print("\nHoldout 환자 분포")
print(holdout_subjects["y"].value_counts().sort_index())

overlap = (
    set(train_subjects["patient_id"])
    & set(holdout_subjects["patient_id"])
)

print("\nTrain–holdout 중복:", len(overlap))

assert train["X"].shape == (5500, 7, 47)
assert holdout["X"].shape == (1440, 7, 47)
assert len(dataset["feature_names"]) == 47
assert len(overlap) == 0

print("\n전처리 데이터 무결성 검사 완료")

Train X: (5500, 7, 47)
Holdout X: (1440, 7, 47)
Feature 수: 47

Train 환자 분포
y
0    84
1    46
2     8
Name: count, dtype: int64

Holdout 환자 분포
y
0    26
1     4
2     2
Name: count, dtype: int64

Train–holdout 중복: 0

전처리 데이터 무결성 검사 완료


In [19]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/"
    "GoogleAI_contest/Jaehwang/3class/lstm_training"
)

CODE_DIR = PROJECT_DIR / "code"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RESULTS_DIR = PROJECT_DIR / "results"

TRAIN_SCRIPT = (
    CODE_DIR / "train_3class_groupcv_holdout.py"
)

DATASET_PATH = (
    PROCESSED_DIR / "lstm_3class_train_holdout.pkl"
)

FAST_RESULT_DIR = (
    RESULTS_DIR / "3class_groupcv_smoke"
)

FULL_RESULT_DIR = (
    RESULTS_DIR / "3class_groupcv_full"
)

print("학습 코드:", TRAIN_SCRIPT.exists())
print("전처리 PKL:", DATASET_PATH.exists())

학습 코드: True
전처리 PKL: True


In [20]:
!python "{TRAIN_SCRIPT}" \
  --data "{DATASET_PATH}" \
  --out "{FAST_RESULT_DIR}" \
  --preset fast \
  --n-splits 4 \
  --epochs 8 \
  --batch-size 128 \
  --patience 3 \
  --seed 42

2026-07-14 14:10:32.564357: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-14 14:10:32.630554: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
3-class patient-group CV training
TensorFlow: {'gpu_count': 1, 'devices': ['/physical_device:GPU:0'], 'mixed_precision_policy': 'mixed_float16'}
Train X: (5500, 7, 47)
Holdout X: (1440, 7, 47)
Train patients: {'Normal': 84, 'MCI': 46, 'Dementia': 8}
Configs: ['rf20_report_backbone']

FOLD 1/4
Train subject distribution: {0: 63, 1: 32, 2: 8}
Validation subject distribution

In [25]:
TRAIN_SCRIPT = (
    CODE_DIR / "train_3class_groupcv_holdout.py"
)

In [29]:
!python "{TRAIN_SCRIPT}" \
  --data "{DATASET_PATH}" \
  --out "{FAST_RESULT_DIR}" \
  --preset fast \
  --n-splits 4 \
  --epochs 8 \
  --batch-size 128 \
  --patience 3 \
  --seed 42

2026-07-14 14:18:51.301019: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-14 14:18:51.367798: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
3-class patient-group CV training
TensorFlow: {'gpu_count': 1, 'devices': ['/physical_device:GPU:0'], 'mixed_precision_policy': 'mixed_float16'}
Train X: (5500, 7, 47)
Holdout X: (1440, 7, 47)
Train patients: {'Normal': 84, 'MCI': 46, 'Dementia': 8}
Configs: ['rf20_report_backbone']

FOLD 1/4
Train subject distribution: {0: 63, 1: 34, 2: 6}
Validation subject distribution

In [31]:
!python "{TRAIN_SCRIPT}" \
  --data "{DATASET_PATH}" \
  --out "{FULL_RESULT_DIR}" \
  --preset full \
  --n-splits 4 \
  --epochs 120 \
  --batch-size 64 \
  --patience 15 \
  --seed 42 \
  --bias-min 0.0 \
  --bias-max 0.0 \
  --bias-step 0.1

2026-07-14 14:25:27.053649: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-14 14:25:27.123051: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
3-class patient-group CV training
TensorFlow: {'gpu_count': 1, 'devices': ['/physical_device:GPU:0'], 'mixed_precision_policy': 'mixed_float16'}
Train X: (5500, 7, 47)
Holdout X: (1440, 7, 47)
Train patients: {'Normal': 84, 'MCI': 46, 'Dementia': 8}
Configs: ['rf20_report_backbone', 'rf30_focal', 'all47_regularized']

FOLD 1/4
Train subject distribution: {0: 63, 1: 34, 2:

In [32]:
import pandas as pd

pred_path = (
    FULL_RESULT_DIR
    / "holdout_subject_predictions.csv"
)

pred = pd.read_csv(pred_path)

class_names = {
    0: "Normal",
    1: "MCI",
    2: "Dementia",
}

pred["true_name"] = pred["y"].map(class_names)
pred["pred_name"] = pred["pred_calibrated"].map(class_names)

display(
    pred[
        [
            "patient_id",
            "true_name",
            "pred_name",
            "p_normal",
            "p_mci",
            "p_dementia",
        ]
    ].sort_values(
        ["y", "p_dementia"],
        ascending=[True, False],
    )
)

KeyError: 'y'